# 00 — Setup: Data + Groups + User mapping

Run this once before the demo. Creates everything the demo depends on:
- **Catalog/schema:** `uc_demo.sample`
- **Tables:** `employees` (1000 rows), `customers` (1000 rows), `user_region_map`
- **Workspace groups (manual, pre-demo):** `admins`, `managers`

Companion notebook `01_demo.py` covers RBAC + ABAC + consumption.

---
## Part A — Synthetic Data
Idempotent — overwrites on re-run.

In [0]:
%pip install faker --quiet
dbutils.library.restartPython()

In [0]:
CATALOG = "uc_demo"
SCHEMA  = "sample"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

In [0]:
from faker import Faker
import random
from datetime import date
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType
)

fake = Faker("da_DK")
Faker.seed(42)
random.seed(42)

def fake_cpr(birthdate):
    """Danish CPR format: DDMMYY-XXXX. Not a real validation, just shape."""
    dd = birthdate.strftime("%d")
    mm = birthdate.strftime("%m")
    yy = birthdate.strftime("%y")
    serial = random.randint(1000, 9999)
    return f"{dd}{mm}{yy}-{serial}"

## employees (1000 rows)
Sensitive columns: `salary`, `cpr`, `email`. ABAC will mask these for non-HR.

In [0]:
DEPARTMENTS = ["Engineering", "Finance", "Sales", "HR", "Marketing"]
COUNTRIES   = ["DK", "SE", "NO", "DE", "US"]

emp_rows = []
for i in range(1, 1001):
    birth = fake.date_of_birth(minimum_age=22, maximum_age=65)
    emp_rows.append(Row(
        emp_id     = i,
        full_name  = fake.name(),
        email      = fake.company_email(),
        cpr        = fake_cpr(birth),
        salary     = round(random.uniform(45_000, 220_000), 2),
        country    = random.choice(COUNTRIES),
        department = random.choice(DEPARTMENTS),
        hire_date  = fake.date_between(start_date=date(2015, 1, 1), end_date=date(2025, 12, 31)),
    ))

emp_schema = StructType([
    StructField("emp_id",     IntegerType(), False),
    StructField("full_name",  StringType(),  False),
    StructField("email",      StringType(),  False),
    StructField("cpr",        StringType(),  False),
    StructField("salary",     DoubleType(),  False),
    StructField("country",    StringType(),  False),
    StructField("department", StringType(),  False),
    StructField("hire_date",  DateType(),    False),
])

(spark.createDataFrame(emp_rows, schema=emp_schema)
      .write.mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{CATALOG}.{SCHEMA}.employees"))

display(spark.table(f"{CATALOG}.{SCHEMA}.employees").limit(5))

## customers (1000 rows)
Sensitive columns: `email`, `date_of_birth`. Region is the key for row-filter policies.

In [0]:
REGIONS = ["EMEA", "AMER", "APAC"]
TIERS   = ["bronze", "silver", "gold", "platinum"]

cust_rows = []
for i in range(1, 1001):
    cust_rows.append(Row(
        customer_id     = i,
        full_name       = fake.name(),
        email           = fake.email(),
        date_of_birth   = fake.date_of_birth(minimum_age=18, maximum_age=85),
        tier            = random.choices(TIERS, weights=[40, 30, 20, 10])[0],
        region          = random.choice(REGIONS),
        lifetime_value  = round(random.uniform(50, 50_000), 2),
    ))

cust_schema = StructType([
    StructField("customer_id",    IntegerType(), False),
    StructField("full_name",      StringType(),  False),
    StructField("email",          StringType(),  False),
    StructField("date_of_birth",  DateType(),    False),
    StructField("tier",           StringType(),  False),
    StructField("region",         StringType(),  False),
    StructField("lifetime_value", DoubleType(),  False),
])

(spark.createDataFrame(cust_rows, schema=cust_schema)
      .write.mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{CATALOG}.{SCHEMA}.customers"))

display(spark.table(f"{CATALOG}.{SCHEMA}.customers").limit(5))

## Sanity check

In [0]:
print("employees:", spark.table(f"{CATALOG}.{SCHEMA}.employees").count())
print("customers:", spark.table(f"{CATALOG}.{SCHEMA}.customers").count())
spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(truncate=False)

---
## Part B — Groups (manual pre-demo step)

Databricks Free Edition has no account console / account API access, so
groups are created manually via the workspace admin UI before the demo.

### Pre-demo checklist
In **Settings → Identity and access → Groups**, create these two groups
and add the corresponding test users as members:

| Group | Intent | Demo behavior |
| --- | --- | --- |
| `admins` | No restrictions | Sees raw rows + raw PII |
| `managers` | Cross-region but PII-masked | Sees all rows, PII columns masked |

There is **no `regular_user` group**. Regular users are recognised by
`current_user()` joined to the `user_region_map` table built in Part C.
Anyone in `user_region_map` but not in either group above is a "regular user."

### Demo identity assignment
| Persona | Email | Group / Map |
| --- | --- | --- |
| Admin | `kristian.johannesen@outlook.dk` | `admins` |
| Manager | `krijztianj@gmail.com` | `managers` |
| Regular user | `krjo@kapacity.dk` | row in `user_region_map` (region=EMEA) |

In [0]:
# Re-declare constants for the demo notebook to import / re-use
CATALOG = "uc_demo"
SCHEMA  = "sample"

DEMO_GROUPS = ["admins", "managers"]

# Sanity check: confirm the groups exist and are visible to this user
groups_in_ws = {row["name"] for row in spark.sql("SHOW GROUPS").collect()}
missing = [g for g in DEMO_GROUPS if g not in groups_in_ws]
if missing:
    print(f"[warn] missing groups (create in UI before running 01_demo): {missing}")
else:
    print(f"[ok] all demo groups present: {DEMO_GROUPS}")

---
## Part C — User → Region mapping table

The ABAC row-filter policy joins this against `current_user()` to decide
which `region` rows a regular user is allowed to see.

**Action required:** replace `<REPLACE_WITH_REAL_USER_EMAIL>` below with
the email of the real Databricks user you'll log in as during the
"regular user" part of the demo.

In [0]:

demo_user = 'kristian.johannesen@twoday.com'

# (email, region) pairs. Add as many rows as you have test users to differentiate.
USER_REGION_MAP = [
    (demo_user, "EMEA"),
]

map_schema = StructType([
    StructField("user_email", StringType(), False),
    StructField("region",     StringType(), False),
])

(spark.createDataFrame(USER_REGION_MAP, schema=map_schema)
      .write.mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{CATALOG}.{SCHEMA}.user_region_map"))

display(spark.table(f"{CATALOG}.{SCHEMA}.user_region_map"))

---
## Cleanup

Tears down the data created above. Groups stay (they were created manually
in the UI; remove them in the UI if you want them gone). The companion
`01_demo.py` has its own cleanup for policies/functions/tags.

In [0]:
%skip
# Drop catalog (and all child tables). Uncomment to run.
spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE")